# RecapAI - Meeting Summarizer & Evaluator
**Course:** CSC 603 - Generative AI | **Team:** Adrian Aquino, Charlie Huynh, Will Brust | **Spring 2026**

## Colab Setup
Run this cell first if you are on Google Colab. It clones the repo and sets up the working directory. Skip if running locally.

In [1]:
# Colab Setup
# Clones the repo if not already present, then pulls the latest.
# Skips everything when running locally.
import os
import shutil

try:
    from google.colab import files, userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    repo       = 'CSC-603-Capstone---Meeting-Summarizer'
    repo_owner = 'SQ-Ghost'
    repo_url   = f'https://github.com/{repo_owner}/{repo}.git'
    os.chdir('/content')

    # guard against nested repo copies left by old buggy runs
    nested_path = os.path.join(repo, repo)
    if os.path.exists(nested_path):
        shutil.rmtree(nested_path)
        print('Cleaned up nested repo copies.')

    # if the repo folder already has a .git dir, just pull; otherwise clone fresh
    if os.path.isdir(os.path.join(repo, '.git')):
        print('Repo found. Pulling latest...')
        os.chdir(repo)
    else:
        if os.path.exists(repo):
            shutil.rmtree(repo)
            print('Removed stale repo folder.')
        os.system(f'git clone {repo_url}')
        os.chdir(repo)
        print('Cloned repo.')

    os.system('git pull')
    print(f'Ready! Working directory: {os.getcwd()}')
else:
    print('Running locally — skipping Colab setup.')


Running locally — skipping Colab setup.


## Install Libraries

In [ ]:
# Install Libraries
%pip install -q torch transformers accelerate huggingface_hub python-dotenv ipywidgets

## Import Libraries

In [ ]:
# Import Libraries
import os
import json
import torch
from transformers import pipeline
from huggingface_hub import login
from dotenv import load_dotenv, set_key

import warnings
warnings.filterwarnings("ignore")
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

# if running from the notebooks folder, step up to project root so all paths work
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

# only import colab files if running on Colab
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print('Done!')

## Hugging Face Login

In [ ]:
# Hugging Face Login
# Colab: reads HF_TOKEN from Secrets panel.
# Local: reads from .env, prompts if missing and saves for next time.
if IN_COLAB:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        print('No HF_TOKEN secret found. Enter your Hugging Face token when prompted, then press Enter.')
        HF_TOKEN = input('HF Token: ')
else:
    load_dotenv()
    HF_TOKEN = os.getenv('HF_TOKEN')

    if not HF_TOKEN:
        print('Enter your Hugging Face token when prompted, then press Enter.')
        HF_TOKEN = input('HF Token: ')
        set_key('.env', 'HF_TOKEN', HF_TOKEN)
        print('Token saved to .env')

login(token=HF_TOKEN)
print('Logged in!')

## Load the Model

In [5]:
# Model Setup
# Uses GPU (float16) when available for speed; falls back to CPU (float32).
MODEL_NAME = 'meta-llama/Llama-3.2-1B-Instruct'

# use GPU if available, otherwise CPU
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using: {DEVICE}')

llm = pipeline(
    task='text-generation',
    model=MODEL_NAME,
    device_map='auto',
    dtype=torch.float16 if DEVICE == 'cuda' else torch.float32,
)

# Remove the model-level max_length cap so max_new_tokens in each
# pipeline call takes full control with no conflict warnings.
llm.model.generation_config.max_length = None

print('Model loaded!')

## Summarize Function

In [ ]:
# Prompt Templates
# ROLE tells the model what it is.
# TASK tells it exactly what to output.
# SummaryEvaluation reads these from this file to stay in sync.
ROLE = "You are an AI assistant that summarizes meeting transcripts into structured JSON."

TASK = """Summarize the following meeting transcript.
Return ONLY a JSON object with exactly these 4 keys:

- "summary": write exactly 2 to 3 sentences. First sentence: what the meeting was about. Second sentence: what was discussed or decided. NEVER write only one sentence — two sentences minimum.
- "decisions": a list of strings, one per real decision made in this transcript (e.g. "Agreed to move the team standup to Wednesdays"). Do NOT copy example text. Only include decisions that actually appear in the transcript. Each item must be a plain string, NOT an object.
- "assigned_tasks": ALWAYS a JSON array starting with [ and ending with ]. Each item must be an object with "who" (first name from the transcript), "what" (the task), and "due" (deadline or "Not specified"). Even if there is only one task, still wrap it in a list: [{"who": "Sarah", "what": "book the venue", "due": "by Friday"}]. Never return a single object without the surrounding list brackets.
- "open_questions": a list of questions that were raised but not resolved in this transcript.

Do not include any explanation or text outside the JSON."""

# sends a single chunk to the model and returns parsed JSON
def summarize_chunk(transcript, max_tokens=1024):
    """Send one transcript chunk to the model and get back parsed JSON."""
    messages = [
        {'role': 'system', 'content': f'{ROLE}'},
        {'role': 'user',   'content': f'{TASK}\n\nTranscript:\n{transcript}'}
    ]

    response = llm(
        messages,
        max_new_tokens=max_tokens,
        do_sample=False,
        repetition_penalty=1.3,  # stops the model from looping the same text
    )

    raw_output = response[0]['generated_text'][-1]['content']
    result = try_parse_json(raw_output)

    # keep only the 4 expected keys — drop anything the model invented (e.g. additionalDecisions)
    result = {k: result.get(k, '' if k == 'summary' else [])
              for k in ('summary', 'decisions', 'assigned_tasks', 'open_questions')}

    # fix: decisions returned as dict instead of list
    if isinstance(result.get('decisions'), dict):
        result['decisions'] = [str(v) for v in result['decisions'].values() if str(v).strip()]

    # fix: flatten nested lists in decisions  [[...]] → [...]
    flat_decisions = []
    for d in (result.get('decisions') or []):
        if isinstance(d, list):
            flat_decisions.extend(d)
        else:
            flat_decisions.append(d)
    # convert any dict decisions to plain strings
    result['decisions'] = [
        d.get('what', '') if isinstance(d, dict) else str(d)
        for d in flat_decisions if d
    ]
    result['decisions'] = [d for d in result['decisions'] if str(d).strip()]

    # fix: assigned_tasks returned as dict instead of list
    if isinstance(result.get('assigned_tasks'), dict):
        result['assigned_tasks'] = [result['assigned_tasks']]

    # fix: flatten nested lists in assigned_tasks  [[...]] → [...]
    flat_tasks = []
    for t in (result.get('assigned_tasks') or []):
        if isinstance(t, list):
            flat_tasks.extend(t)
        else:
            flat_tasks.append(t)
    result['assigned_tasks'] = flat_tasks

    # ensure every task has all 3 required fields — add defaults for missing ones
    fixed_tasks = []
    for t in result['assigned_tasks']:
        if isinstance(t, dict):
            t.setdefault('who', 'Unassigned')
            t.setdefault('what', '')
            t.setdefault('due', 'Not specified')
            fixed_tasks.append(t)
        elif isinstance(t, str) and t.strip():
            fixed_tasks.append({'who': 'Unassigned', 'what': t.strip(), 'due': 'Not specified'})
    result['assigned_tasks'] = fixed_tasks

    # filter out tasks where who or what are empty — they are useless
    result['assigned_tasks'] = [
        t for t in result['assigned_tasks']
        if isinstance(t, dict)
        and (t.get('who') or '').strip() not in ('', 'Unassigned', 'Not specified')
        and (t.get('what') or '').strip()
    ]

    # deduplicate and cap each list — guards against repetition loops
    for key in ('decisions', 'assigned_tasks', 'open_questions'):
        if isinstance(result.get(key), list):
            result[key] = removedupe(result[key])[:8]

    # --- guarantee no field is blank when saved to disk ---

    # summary: if it's a code blob or empty, replace with a readable fallback
    s = result.get('summary', '').strip()
    if not s or s.startswith('{') or s.startswith('```'):
        result['summary'] = '(Summary could not be extracted from this transcript.)'

    # decisions: keep as [] if none found — that is a valid value
    if not isinstance(result.get('decisions'), list):
        result['decisions'] = []

    # assigned_tasks: each task must have who, what, due as non-blank strings
    cleaned = []
    for t in result.get('assigned_tasks', []):
        if isinstance(t, dict):
            cleaned.append({
                'who':  (t.get('who')  or 'Unassigned').strip(),
                'what': (t.get('what') or '(no description)').strip(),
                'due':  (t.get('due')  or 'Not specified').strip(),
            })
    result['assigned_tasks'] = cleaned

    # open_questions: keep as [] if none found — that is a valid value
    if not isinstance(result.get('open_questions'), list):
        result['open_questions'] = []

    return result


# MOCK_PROMPT uses {topic} so each generated transcript covers a different subject
MOCK_PROMPT = """Write a realistic office meeting about {topic} as one single unbroken paragraph of natural speech. No line breaks. No formatting.

NEVER do any of these:
- Never write a name followed by a colon (e.g. never write Mike: or Sarah: or John:)
- Never put each person on their own line
- Never add bullet points, numbers, or headers
- Never add quotation marks around what people say
- Never add metadata like date, attendees, or meeting title

The output must be one continuous block of plain text where names appear naturally inside sentences only, like: hey Mike what do you think about that, yeah I agree we should go with the cheaper option, okay so who is handling the follow-up email, I can take care of that by next Monday no problem.

Include exactly 3 people by first name only (e.g. Sarah, David, Tom). Use each name naturally inside sentences. Include at least one decision, one task assigned to a specific named person with a real deadline, and one unresolved question."""


# list of realistic everyday work meeting topics
MEETING_TOPICS = [
    'planning the company holiday party',
    'deciding on a new office coffee and snack budget',
    'updating the team vacation and time-off policy',
    'organizing the all-hands presentation for next month',
    'reviewing the employee onboarding checklist',
    'planning a team lunch outing',
    'discussing the new employee parking situation',
    'setting up weekly team check-in meetings',
    'reviewing feedback from the last team survey',
    'planning the office move to a new floor',
    'updating the work-from-home schedule for the quarter',
    'discussing dress code for the upcoming client visit',
    'organizing volunteer day for the team',
    'planning training sessions for new software tools',
    'reviewing the team meeting schedule and cutting unnecessary ones',
    'discussing how to handle meeting overruns and late starts',
    'planning the end-of-year team retrospective',
    'organizing a welcome event for new hires',
    'deciding on a new project management tool for the team',
    'reviewing the shared calendar and booking process for meeting rooms',
]

print('Prompt templates loaded!')

## JSON Parser with Fallback

In [ ]:
import re
import ast

def try_parse_json(raw_output):
    """Parses model output as JSON. Tries direct parse, then asks model to repair, then falls back.

    :param raw_output: Raw text returned by the model.
    :type raw_output: string
    :returns: Parsed JSON or fallback dictionary with raw text under 'summary'.
    :return type: dictionary
    """
    # safety: if output is abnormally long it is probably a repetition loop — truncate early
    if len(raw_output) > 6000:
        raw_output = raw_output[:6000]

    # strip markdown code fences before any parsing attempt — model often wraps JSON in ```json
    fence = re.search(r'```(?:json)?\s*\n?([\s\S]*?)\n?\s*```', raw_output)
    if fence:
        raw_output = fence.group(1).strip()

    # strategy 1 — find the first { } block and parse as standard JSON
    try:
        start  = raw_output.index('{')
        end    = raw_output.rindex('}') + 1
        parsed = json.loads(raw_output[start:end])
        return parsed
    except (ValueError, json.JSONDecodeError):
        pass

    # strategy 1b — try ast.literal_eval for Python-style single-quoted output
    try:
        start  = raw_output.index('{')
        end    = raw_output.rindex('}') + 1
        parsed = ast.literal_eval(raw_output[start:end])
        if isinstance(parsed, dict):
            return parsed
    except Exception:
        pass

    # strategy 2 — send broken output back to the model and ask it to fix
    try:
        repair_messages = [
            {'role': 'system', 'content': 'You fix broken JSON. Return only valid JSON and nothing else.'},
            {'role': 'user',   'content': f'Fix this JSON:\n\n{raw_output[:2000]}'}
        ]
        repair_response = llm(repair_messages, max_new_tokens=512, do_sample=False)
        repaired        = repair_response[0]['generated_text'][-1]['content']
        # strip fences from repair output too
        fence2 = re.search(r'```(?:json)?\s*\n?([\s\S]*?)\n?\s*```', repaired)
        if fence2:
            repaired = fence2.group(1).strip()
        start           = repaired.index('{')
        end             = repaired.rindex('}') + 1
        parsed          = json.loads(repaired[start:end])
        return parsed
    except (ValueError, json.JSONDecodeError):
        pass

    # strategy 3 — give up parsing, return raw text so the UI still shows something
    return {
        'summary': raw_output[:500],  # cap so summary field is not a giant blob
        'decisions': [],
        'assigned_tasks': [],
        'open_questions': ['[Could not parse output - see summary for raw text]']
    }

print('Function defined!')

## Validate Output

In [ ]:
def validate_output(result):
    """Checks that all 4 required keys exist in the summary output.

    :param result: Parsed output from summarize_meeting().
    :type result: dictionary
    :returns: True if all sections present, False if any are missing.
    :return type: boolean
    """
    required    = ['summary', 'decisions', 'assigned_tasks', 'open_questions']
    all_present = True

    for section in required:
        if section not in result:
            print(f"WARNING: Missing -> '{section}'")
            all_present = False
        else:
            print(f"OK: '{section}'")

    print('\nAll good!' if all_present else '\nSome sections missing. Check the prompt.')
    return all_present

print('Function defined!')

## Chunking (for long transcripts)

In [ ]:
# max characters sent to the model per chunk
CHUNK_SIZE = 1500

def chunk_transcript(transcript, max_chars=CHUNK_SIZE):
    """Splits a long transcript into sentence-boundary chunks.

    :param transcript: The full transcript text.
    :type transcript: string
    :param max_chars: Max characters per chunk.
    :type max_chars: integer
    :returns: List of transcript chunks.
    :return type: list of strings
    """
    sentences = transcript.replace('\n', ' ').split('. ')
    chunks  = []
    current = ''

    for sentence in sentences:
        segment = sentence + '. '
        if len(current) + len(segment) > max_chars and current:
            chunks.append(current.strip())
            current = segment
        else:
            current += segment

    if current.strip():
        chunks.append(current.strip())

    return chunks


def removedupe(items):
    """Removes duplicates from a list while keeping the original order.

    :param items: Strings or dictionaries to deduplicate.
    :type items: list
    :returns: Deduplicated list in original order.
    :return type: list
    """
    seen   = set()
    result = []

    for item in items:
        key = json.dumps(item, sort_keys=True) if isinstance(item, dict) else str(item)
        if key not in seen:
            seen.add(key)
            result.append(item)

    return result


def merge_results(results):
    """Combines per-chunk JSON outputs into one result and removes duplicates.

    :param results: List of JSON outputs from summarize_chunk().
    :type results: list of dictionaries
    :returns: Single merged JSON output.
    :return type: dictionary
    """
    merged = {
        'summary':        '',
        'decisions':      [],
        'assigned_tasks': [],
        'open_questions': []
    }

    summaries = []
    for r in results:
        if r.get('summary'):
            summaries.append(r['summary'])
        merged['decisions']      += r.get('decisions', [])
        merged['assigned_tasks'] += r.get('assigned_tasks', [])
        merged['open_questions'] += r.get('open_questions', [])

    merged['summary']        = ' '.join(summaries)
    merged['decisions']      = removedupe(merged['decisions'])
    merged['assigned_tasks'] = removedupe(merged['assigned_tasks'])
    merged['open_questions'] = removedupe(merged['open_questions'])

    return merged


def summarize_meeting(transcript, max_chars=CHUNK_SIZE):
    """Main entry point. Summarizes a transcript, chunking it first if it is too long.

    :param transcript: The full transcript text.
    :type transcript: string
    :param max_chars: Max characters per chunk.
    :type max_chars: integer
    :returns: Merged JSON output.
    :return type: dictionary
    """
    if len(transcript) <= max_chars:
        return summarize_chunk(transcript)

    chunks = chunk_transcript(transcript, max_chars)

    if len(chunks) == 1:
        return summarize_chunk(chunks[0])

    results = []
    for i, chunk in enumerate(chunks):
        results.append(summarize_chunk(chunk))

    return merge_results(results)


print('Chunking functions defined!')

## Input Transcript
Choose how to provide your transcript: paste text, upload a file, or generate a mock transcript using AI.

In [ ]:
print('How would you like to input the transcript?')
print('1. Paste plain text')
print('2. Upload a .txt file')
print('3. Generate mock transcripts using AI')

while True:
    choice = input('Enter choice (1/2/3): ').strip()
    if choice in ('1', '2', '3'):
        break
    print(f'"{choice}" is not valid — please enter 1, 2, or 3.')

os.makedirs('Transcripts', exist_ok=True)
os.makedirs('Summaries', exist_ok=True)

def _clear_old_transcripts_and_summaries():
    for f in os.listdir('Transcripts'):
        if f.endswith('.txt'):
            os.remove(f'Transcripts/{f}')
    for f in os.listdir('Summaries'):
        if f.startswith('Transcript-') and f.endswith('_summary.json'):
            os.remove(f'Summaries/{f}')
    print('Cleared old transcripts and summaries.')

def _save_and_summarize(transcript_text, index=1):
    t_path = f'Transcripts/Transcript-{index:02d}.txt'
    with open(t_path, 'w', encoding='utf-8') as f:
        f.write(transcript_text)
    print(f'Saved transcript → {t_path}')
    print('Summarizing...')
    result = summarize_meeting(transcript_text)
    s_path = f'Summaries/Transcript-{index:02d}_summary.json'
    with open(s_path, 'w', encoding='utf-8') as f:
        json.dump(result, f, indent=2)
    print(f'Saved summary   → {s_path}')
    return result

# choice 1: paste text directly
if choice == '1':
    print('Paste your transcript below, then press Enter.')
    transcript_text = input('Transcript: ')
    _clear_old_transcripts_and_summaries()
    _save_and_summarize(transcript_text)
    TRANSCRIPT = ''

# choice 2: upload or path to a .txt file
elif choice == '2':
    if IN_COLAB:
        uploaded = files.upload()
        filename = list(uploaded.keys())[0]
        transcript_text = uploaded[filename].decode('utf-8')
        print(f'Uploaded: {filename}')
    else:
        path = input('Enter the full path to your .txt file: ').strip()
        with open(path, 'r', encoding='utf-8') as f:
            transcript_text = f.read()
        print(f'Loaded: {path}')
    _clear_old_transcripts_and_summaries()
    _save_and_summarize(transcript_text)
    TRANSCRIPT = ''

# choice 3: generate mock transcripts using the LLM
elif choice == '3':
    count = int(input('How many transcripts to generate? (1-20): ').strip())
    count = max(1, min(20, count))
    _clear_old_transcripts_and_summaries()
    print(f'Generating {count} transcript(s)...\n')
    for i in range(1, count + 1):
        # pick a different topic for each transcript so they are all unique
        topic = MEETING_TOPICS[(i - 1) % len(MEETING_TOPICS)]
        print(f'Generating transcript {i} of {count} (topic: {topic})...')
        messages = [
            {'role': 'system', 'content': 'You generate realistic meeting transcripts.'},
            {'role': 'user',   'content': MOCK_PROMPT.format(topic=topic)}
        ]
        response       = llm(messages, max_new_tokens=800, do_sample=True, temperature=0.9)
        transcript_txt = response[0]['generated_text'][-1]['content']
        t_path         = f'Transcripts/Transcript-{i:02d}.txt'
        with open(t_path, 'w', encoding='utf-8') as f:
            f.write(transcript_txt)
        print(f'Saved: {t_path}')
    print(f'\nAll {count} transcripts saved!')
    print(f'\nSummarizing all {count} transcripts...')
    for i in range(1, count + 1):
        with open(f'Transcripts/Transcript-{i:02d}.txt', 'r', encoding='utf-8') as f:
            transcript = f.read()
        print(f'\n--- Transcript {i} of {count} ---')
        result  = summarize_meeting(transcript)
        s_path  = f'Summaries/Transcript-{i:02d}_summary.json'
        with open(s_path, 'w', encoding='utf-8') as f:
            json.dump(result, f, indent=2)
        print(f'Saved: {s_path}')
    print(f'\nDone! All {count} transcripts summarized and saved to Summaries/')
    TRANSCRIPT = ''

## Evaluation Rubric and Functions
Scores each summary against the original transcript using the rubric from our proposal:
- **Clarity** (1-5): Is the summary clear and easy to understand?
- **Faithfulness** (1-5): Does it only include facts from the transcript?
- **Decisions** (1-5): Are all decisions correctly captured?
- **Assigned Tasks** (1-5): Are tasks assigned to the right people with correct details?

In [ ]:
# Evaluation Rubric and Functions
# scores each summary using our proposal rubric
# uses rule-based checks so scores are accurate and varied

def evaluate(transcript, summary):
    """Score a summary against the original transcript using the rubric."""
    scores = {}
    issues = []

    # clarity (1-5): is the summary clear and readable?
    s = summary.get('summary', '')
    clarity = 1
    if s and s.strip():
        # detect parse failure — a real summary should not start with backticks or {
        if s.strip().startswith('```') or s.strip().startswith('{'):
            clarity = 1
            issues.append('Summary contains raw JSON or code instead of plain text')
        else:
            clarity = 2
            import re
            sentences = [x.strip() for x in re.split(r'[.!?]+', s) if x.strip()]
            if len(sentences) >= 2: clarity = 3
            if len(sentences) >= 2 and len(s) >= 50: clarity = 4
            if len(sentences) >= 2 and len(s) >= 80 and len(s) <= 500: clarity = 5
            if len(sentences) < 2:
                issues.append('Summary is only one sentence, should be 2-3')
            if len(s) > 500:
                issues.append('Summary is too long (over 500 chars)')
                clarity = min(clarity, 3)
    else:
        issues.append('Summary is empty')
    scores['clarity'] = clarity

    # faithfulness (1-5): only facts from transcript?
    faithfulness = 3
    transcript_lower = transcript.lower()

    # robustness: handle assigned_tasks returned as dict instead of list
    raw_tasks = summary.get('assigned_tasks', [])
    if isinstance(raw_tasks, dict):
        raw_tasks = [raw_tasks] if (raw_tasks.get('who') or raw_tasks.get('what')) else []
    tasks = raw_tasks

    names_in_tasks = []
    for t in tasks:
        if isinstance(t, dict):
            name = (t.get('who') or '').strip()
            if name and name.lower() not in ('unassigned', 'not specified'):
                names_in_tasks.append(name)

    names_found, names_missing = 0, []
    for name in names_in_tasks:
        # try full name first, then each part — handles "Mike Johnson" vs "Mike"
        parts = [p for p in name.lower().split() if len(p) > 2]
        if name.lower() in transcript_lower or any(p in transcript_lower for p in parts):
            names_found += 1
        else:
            names_missing.append(name)

    if names_in_tasks:
        ratio = names_found / len(names_in_tasks)
        if ratio == 1.0: faithfulness = 5
        elif ratio >= 0.5: faithfulness = 4
        else:
            faithfulness = 2
            issues.append(f'Names not found in transcript: {names_missing}')
    else:
        faithfulness = 3  # no named assignees — neutral

    # use content words only (length > 3, skip common stop words)
    stop_words = {
        'the','a','an','is','are','was','were','it','to','of','in','and','or','for','on','at',
        'by','be','that','this','with','as','from','we','i','you','he','she','they','not','will',
        'have','has','had','do','does','did','but','if','so','also','just','their','our','about','been'
    }
    # skip the overlap check if the summary is a parse failure
    if not (s.strip().startswith('```') or s.strip().startswith('{')):
        summary_words    = {w for w in s.lower().split() if len(w) > 3 and w not in stop_words}
        transcript_words = {w for w in transcript_lower.split() if len(w) > 3 and w not in stop_words}
        overlap = len(summary_words & transcript_words)
        if overlap < 3:
            faithfulness = max(1, faithfulness - 1)
            issues.append('Very little content-word overlap between summary and transcript')
    scores['faithfulness'] = faithfulness

    # decisions (1-5): are decisions correctly captured?
    decisions = summary.get('decisions', [])
    dec_score = 1
    if isinstance(decisions, list) and len(decisions) > 0:
        dec_score = 3
        all_strings = all(isinstance(d, str) for d in decisions)
        if all_strings:
            dec_score = 4
            if len(decisions) >= 2: dec_score = 5
        else:
            issues.append('Some decisions are objects instead of strings')
            has_content = any(
                (isinstance(d, dict) and (d.get('what') or '').strip())
                or (isinstance(d, str) and d.strip())
                for d in decisions
            )
            if has_content: dec_score = 3
    else:
        if any(w in transcript_lower for w in ['decided','agreed','decision','go with','approved']):
            issues.append('Transcript mentions decisions but none were extracted')
            dec_score = 1
        else:
            dec_score = 3
            issues.append('No decisions found (may be correct if none were made)')
    scores['decisions'] = dec_score

    # assigned_tasks (1-5): correct people and details?
    task_score = 1
    if isinstance(tasks, list) and len(tasks) > 0:
        task_score = 2
        full_tasks    = 0  # tasks with who + what + due
        partial_tasks = 0  # tasks with at least who + what

        for t in tasks:
            if isinstance(t, dict):
                has_who  = (t.get('who')  or '').strip() not in ('','Unassigned','Not specified')
                has_what = bool((t.get('what') or '').strip())
                has_due  = (t.get('due')  or '').strip() not in ('','Not specified','N/A')
                if has_who and has_what and has_due: full_tasks += 1
                if has_who and has_what: partial_tasks += 1
                if not has_who: issues.append(f'Task missing assignee: {(t.get("what") or "?")[:40]}')
                if not has_what: issues.append('Task has no description')
                if has_who and has_what and not has_due: issues.append(f'Task for {t.get("who","?")} is missing a due date')
            else:
                issues.append('Task is a string instead of object with who/what/due')

        if full_tasks >= 2:
            task_score = 5
        elif full_tasks >= 1:
            task_score = 4
        elif partial_tasks >= 2:
            task_score = 4
        elif partial_tasks >= 1:
            task_score = 3
    else:
        if any(w in transcript_lower for w in ['handle','take care','assigned','responsible','by friday','by monday','by thursday']):
            issues.append('Transcript mentions tasks but none were extracted')
            task_score = 1
        else:
            task_score = 3
            issues.append('No tasks found (may be correct)')
    scores['assigned_tasks'] = task_score

    return {'scores': scores, 'issues': issues}


def display_results(result):
    """Print the evaluation scores and any issues found."""
    if 'scores' not in result:
        print('No scores available.')
        return

    scores = result.get('scores', {})
    print('=' * 40)
    print('EVALUATION SCORES')
    print('=' * 40)
    print('  Clarity:        ' + str(scores.get('clarity',        'N/A')) + ' / 5')
    print('  Faithfulness:   ' + str(scores.get('faithfulness',   'N/A')) + ' / 5')
    print('  Decisions:      ' + str(scores.get('decisions',      'N/A')) + ' / 5')
    print('  Assigned Tasks: ' + str(scores.get('assigned_tasks', 'N/A')) + ' / 5')
    print()

    issues = result.get('issues', [])
    if issues:
        print('ISSUES FOUND')
        print('-' * 40)
        for i, issue in enumerate(issues, 1):
            print('  ' + str(i) + '. ' + str(issue))
        print()


print('Evaluation functions defined!')

## Evaluate Transcripts & Improve Prompts
Scores every transcript/summary pair in `Transcripts/` and `Summaries/`, then gives exact copy-paste instructions for fixing the prompts.
- Per-transcript results saved to `Evaluations/TranscriptEvaluations/` (cleared each run)
- Numbered run summaries saved to `Evaluations/EvaluationSummaries/` (kept across runs)

In [ ]:
# Evaluate Transcripts & Improve Prompts
import os
import json
from collections import Counter

W       = 68
CW      = (16, 7, 12, 9, 14)
TBL_DIV = '  ' + '─' * (sum(CW) + 2 * (len(CW) - 1))

def _bar(score, total=5, width=10):
    v      = float(score or 0)
    pct    = int(v / total * 100)
    filled = int(v / total * width)
    return f'{pct:3d}%|{"█" * filled}{" " * (width - filled)}|'

LABELS = {
    'clarity':        'Clarity',
    'faithfulness':   'Faithfulness',
    'decisions':      'Decisions',
    'assigned_tasks': 'Assigned Tasks',
}
SCORE_KEYS = ['clarity', 'faithfulness', 'decisions', 'assigned_tasks']

EVAL_TRANSCRIPTS = 'Evaluations/TranscriptEvaluations'
EVAL_SUMMARIES   = 'Evaluations/EvaluationSummaries'

# few-shot examples so the model learns what good fixes look like
FEW_SHOT_FIXES = """\
Problem: Summary is only one sentence, should be 2-3
Fix: - The summary must be exactly 2-3 sentences. First sentence: what the meeting was about. Second sentence: what was discussed or decided.

Problem: Some decisions are objects instead of strings
Fix: - Each decision must be a plain string, for example "Agreed to push the deadline to Friday". Never use a dictionary or object.

Problem: Task missing assignee
Fix: - Every task must include the exact first name of the person responsible, taken directly from the transcript.

Problem: Summary is too long (over 500 chars)
Fix: - Keep the summary under 500 characters. One or two sentences is enough — cut any repeated detail."""

# collect all transcript/summary pairs
all_items = []
if os.path.exists('Summaries') and os.path.exists('Transcripts'):
    for fname in sorted(os.listdir('Summaries')):
        if not fname.endswith('_summary.json'):
            continue
        base            = fname.replace('_summary.json', '')
        transcript_path = f'Transcripts/{base}.txt'
        if os.path.exists(transcript_path):
            with open(transcript_path, 'r', encoding='utf-8') as f:
                transcript = f.read()
            with open(f'Summaries/{fname}', 'r', encoding='utf-8') as f:
                summary = json.load(f)
            all_items.append((base, transcript, summary))

if not all_items:
    print('No transcript/summary pairs found.')
    print('Run the Input Transcript section above first.')
else:
    os.makedirs(EVAL_TRANSCRIPTS, exist_ok=True)
    os.makedirs(EVAL_SUMMARIES,   exist_ok=True)

    for f in os.listdir(EVAL_TRANSCRIPTS):
        if f.endswith('.json'):
            os.remove(f'{EVAL_TRANSCRIPTS}/{f}')

    existing = [f for f in os.listdir(EVAL_SUMMARIES) if f.endswith('_summary.json')]
    run_number = len(existing) + 1

    print(f'Evaluating {len(all_items)} transcript(s).  Run #{run_number:03d}\n')

    all_scores      = []
    all_real_issues = []

    for label, transcript, summary in all_items:
        print('=' * W)
        print(f'  {label}')
        print('=' * W)

        result = evaluate(transcript, summary)
        scores = result.get('scores', {})
        issues = result.get('issues', [])

        for k in SCORE_KEYS:
            v = scores.get(k, 0)
            print(f'  {LABELS[k]:<16} {_bar(v)}  {v} / 5')

        real_issues = [i for i in issues if 'may be correct' not in str(i).lower()]
        if real_issues:
            print()
            print('  Found issues:')
            for issue in real_issues:
                print(f'    - {issue}')

        out_name  = label + '_evaluation.json'
        eval_data = {'file': label, 'summary': summary, 'evaluation': result}
        with open(f'{EVAL_TRANSCRIPTS}/{out_name}', 'w') as f:
            json.dump(eval_data, f, indent=2)
        print(f'\n  Saved ----> {EVAL_TRANSCRIPTS}/{out_name}\n')

        if 'scores' in result:
            all_scores.append({'file': label, **result['scores']})
        all_real_issues.extend(real_issues)

    # overall results table
    print('\n' + '=' * W)
    print('  OVERALL RESULTS')
    print('=' * W)
    print(f'  {"File":<{CW[0]}}'
          f'  {"Clarity":>{CW[1]}}'
          f'  {"Faithfulness":>{CW[2]}}'
          f'  {"Decisions":>{CW[3]}}'
          f'  {"Assigned Tasks":>{CW[4]}}')
    print(TBL_DIV)
    for s in all_scores:
        print(f'  {s["file"]:<{CW[0]}}'
              f'  {s.get("clarity",       "?"):>{CW[1]}}'
              f'  {s.get("faithfulness",  "?"):>{CW[2]}}'
              f'  {s.get("decisions",     "?"):>{CW[3]}}'
              f'  {s.get("assigned_tasks","?"):>{CW[4]}}')
    print(TBL_DIV)

    averages = {}
    for k in SCORE_KEYS:
        vals = [s[k] for s in all_scores if k in s and isinstance(s[k], (int, float))]
        averages[k] = round(sum(vals) / len(vals), 2) if vals else None

    print()
    print('  Averages:')
    for k in SCORE_KEYS:
        v     = averages.get(k)
        v_str = str(v) if v is not None else 'N/A'
        print(f'    {LABELS[k]:<16} {_bar(v)}  {v_str} / 5')

    run_summary = {
        'run':             run_number,
        'prompt_role':     ROLE,
        'prompt_task':     TASK,
        'items_evaluated': len(all_items),
        'scores':          all_scores,
        'averages':        averages,
    }
    summary_name = f'run_{run_number:03d}_summary.json'
    with open(f'{EVAL_SUMMARIES}/{summary_name}', 'w') as f:
        json.dump(run_summary, f, indent=2)
    print(f'\n  Summary saved ----> {EVAL_SUMMARIES}/{summary_name}')

    issue_counts  = Counter(all_real_issues)
    unique_issues = list(dict.fromkeys(all_real_issues))

    print('\n' + '=' * W)
    print('  ISSUES FOUND ACROSS ALL EVALUATIONS')
    print('=' * W)
    if issue_counts:
        for issue, count in issue_counts.most_common():
            tag = f'[{count}x]' if count > 1 else '      '
            print(f'  {tag}  {issue}')
    else:
        print('  None — all transcripts evaluated cleanly.')

    if not unique_issues:
        print('\n' + '=' * W)
        print('  No prompt improvements needed — all scores look good.')
        print('=' * W)
    else:
        n              = len(unique_issues)
        issues_labeled = '\n'.join(f'Problem: {i}' for i in unique_issues)

        fix_messages = [
            {'role': 'system', 'content': (
                f'You improve AI prompts. '
                f'For each Problem below, write one Fix starting with "- ". '
                f'Be specific and direct. '
                f'Write exactly {n} Fix(es), one per Problem. No extra text.'
            )},
            {'role': 'user', 'content': (
                f'Here are examples of good fixes:\n\n'
                f'{FEW_SHOT_FIXES}\n\n'
                f'---\n\n'
                f'Now write fixes for these {n} problem(s):\n\n'
                f'{issues_labeled}\n\n'
                f'Write exactly {n} Fix line(s), one per Problem:'
            )}
        ]

        print('\n  Generating prompt improvements...')
        fix_response = llm(fix_messages, max_new_tokens=300, do_sample=True,
                           temperature=0.7, repetition_penalty=1.2)
        raw_fixes    = fix_response[0]['generated_text'][-1]['content'].strip()

        # extract fix lines — accept "- text" or "Fix: text" formats
        fix_lines = []
        for line in raw_fixes.splitlines():
            line = line.strip()
            if line.startswith('-') and len(line) > 10:
                fix_lines.append(line)
            elif line.lower().startswith('fix:'):
                content = line[4:].strip().lstrip('-').strip()
                if len(content) > 10:
                    fix_lines.append('- ' + content)
        fix_lines = fix_lines[:n]

        # if model produced nothing usable, skip — don't append garbage to the prompt
        ai_fixes = '\n'.join(fix_lines)

        # only add rules to the task prompt if we got clean fix lines
        if ai_fixes:
            added_rules = '\n\nAdditional rules based on evaluation:\n' + ai_fixes
            new_task    = TASK.rstrip() + added_rules
        else:
            new_task = TASK

        print('\n' + '=' * W)
        print('  PROMPT IMPROVEMENTS')
        print('=' * W)
        print()
        if ai_fixes:
            for line in ai_fixes.splitlines():
                print(f'  {line}')
        else:
            print('  (Model did not produce usable fix suggestions this run.)')

        print('\n' + '=' * W)
        print('  WHERE TO APPLY')
        print('=' * W)
        print()
        print('  notebooks/RecapAI.ipynb  ->  "Summarize Function" cell')
        print('  Replace the entire TASK = """...""" block with:')
        print()
        print('  ' + '─' * (W - 2))
        print(f'  TASK = """{new_task}"""')
        print('  ' + '─' * (W - 2))
        print()
        print('  Then: delete Summaries/, re-run Input Transcript, and re-run this cell.')
        print('=' * W)